In [14]:
import cv2
import typing
import numpy as np
import matplotlib.pyplot as plt

from mltu.inferenceModel import OnnxInferenceModel
from mltu.utils.text_utils import ctc_decoder
from mltu.configs import BaseModelConfigs

In [15]:
import os
from datetime import datetime

from mltu.configs import BaseModelConfigs


class ModelConfigs(BaseModelConfigs):
    def __init__(self):
        super().__init__()
        self.model_path = os.path.join("../models/", datetime.strftime(datetime.now(), "%Y%m%d%H%M"))
        self.vocab = ""
        self.height = 32
        self.width = 128
        self.max_text_length = 0
        self.batch_size = 64
        self.learning_rate = 0.002
        self.train_epochs = 1000

In [16]:
class ImageToWordModel(OnnxInferenceModel):

    def predict(self, image):
        image = cv2.resize(
            image,
            (self.input_shape[1], self.input_shape[0])
        )

        image_pred = np.expand_dims(image, axis=0).astype(np.float32)

        preds = self.model.run(
            [self.output_name],
            {self.input_name: image_pred}
        )[0]

        return ctc_decoder(preds, VOCAB)[0]

In [20]:
import yaml
import cv2

with open("../models/202301131202/configs.yaml", "r") as f:
    configs = yaml.safe_load(f)

VOCAB = configs["vocab"]

model = ImageToWordModel(
    model_path="../models/202301131202/model.onnx"
)

image = cv2.imread(r"C:\Users\User\Downloads\chek2.jpg")

image = cv2.resize(
    image,
    (model.input_shape[1], model.input_shape[0])
)

image_pred = np.expand_dims(image, axis=0).astype(np.float32)

preds = model.model.run(
    [model.output_name],
    {model.input_name: image_pred}
)[0]

prediction = ctc_decoder(preds, VOCAB)[0]

print("Prediction:", prediction)

Prediction: rncnn-- -n me n-n


In [9]:
class ImageToWordModel(OnnxInferenceModel):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def predict(self, image):
        image = cv2.resize(
            image,
            (self.input_shape[1], self.input_shape[0])
        )
    
        image_pred = np.expand_dims(image, axis=0).astype(np.float32)
    
        preds = self.model.run(
            [self.output_name],
            {self.input_name: image_pred}
        )[0]
    
        return ctc_decoder(preds, self.metadata["vocab"])[0]

if __name__ == "__main__":
    import pandas as pd
    from tqdm import tqdm

    model = ImageToWordModel(model_path="../models/202301131202/model.onnx")

    df = pd.read_csv("../models/202301131202/val.csv").values.tolist()

    accum_cer = []
    for image_path, label in tqdm(df):
        image = cv2.imread(image_path.replace("\\", "/"))

        prediction_text = model.predict(image)

        cer = get_cer(prediction_text, label)
        print(f"Image: {image_path}, Label: {label}, Prediction: {prediction_text}, CER: {cer}")

        accum_cer.append(cer)

    print(f"Average CER: {np.average(accum_cer)}")

  0%|                                                                                          | 0/301 [00:00<?, ?it/s]


error: OpenCV(4.8.1) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4062: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'


In [9]:
model = ImageToWordModel(
    model_path="../models/202301131202/model.onnx"
)

print(dir(model))

['__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'default_model_name', 'force_cpu', 'input_name', 'input_shape', 'metadata', 'model', 'model_path', 'output_name', 'predict']


In [13]:
print("Input shape:", model.input_shape)
print("Input name:", model.input_name)
print("Output name:", model.output_name)

Input shape: [96, 1408, 3]
Input name: input
Output name: output


In [15]:
print(df[0])


['Datasets/IAM_Sentences/sentences/g06/g06-037r/g06-037r-s03-04.png', 'the slightest effect .']


In [16]:
import os
print(os.getcwd())

E:\Codes\Projects\Omni-Text\backend\ML\SentenceToString\train


In [17]:
import os

image_path = df[0][0]

print("Path:", image_path)
print("Exists:", os.path.exists(image_path))
print("Absolute:", os.path.abspath(image_path))

Path: Datasets/IAM_Sentences/sentences/g06/g06-037r/g06-037r-s03-04.png
Exists: False
Absolute: E:\Codes\Projects\Omni-Text\backend\ML\SentenceToString\train\Datasets\IAM_Sentences\sentences\g06\g06-037r\g06-037r-s03-04.png


In [18]:
import os

for root, dirs, files in os.walk("E:/"):
    if "g06-037r-s03-04.png" in files:
        print(os.path.join(root, "g06-037r-s03-04.png"))
        break

In [11]:
print(model.metadata)

{}


In [21]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42
)

print(len(train_df))
print(len(val_df))

270
31


TypeError: list indices must be integers or slices, not str